In [7]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os

print("Files Python can see:", os.listdir('.'))
from google.colab import drive # type: ignore
drive.mount(r'/content/drive/')

# Important Parameters
learning_rate = 0.1
epochs = 25

# Defining a neural network
class MySimpleNN():
    def __init__(self, x):
        self.weights = torch.rand(x.shape[1], 1, dtype=torch.float64, requires_grad=True)
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    
    def forward_pass(self, x):
        z = torch.matmul(x, self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

    # Defining the loss function
    def loss_function(self, y_pred, y):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

        # FIXED: Enclosed the entire binary cross entropy expression inside torch.mean()
        loss = -torch.mean(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred))
        return loss

# Load Data
df = pd.read_csv(r"/content/drive/MyDrive/DL_CSV/breast-cancer.csv")

# FIXED: Saved the dataframe modification back to df so 'id' is removed
df = df.drop(['id'], axis='columns', errors='ignore')

Y = df['diagnosis']
X = df.drop(['diagnosis'], axis='columns')

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)

# Scaling the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Label Encoding
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# Converting Numpy Arrays to tensors
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train).unsqueeze(1) #unsqueese is used to fix the dimension mismatch between y_pred and y_train_tensor as y_pred was of shape[455,1] while y_train is of [455]. Unsqueeze adds a new dimension to the tensor making the size[455,1]
y_test_tensor = torch.from_numpy(y_test).unsqueeze(1)

# Main Function
model = MySimpleNN(X_train_tensor)

for i in range(epochs):
    y_pred = model.forward_pass(X_train_tensor)
    print("For epoch number:", i)
    print("Y_preds", y_pred)
    
    loss = model.loss_function(y_pred, y_train_tensor)
    print("Loss=", loss.item())
    loss.backward()

    # Parameter Update
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    model.weights.grad.zero_()
    model.bias.grad.zero_()

# Model Evaluation
print("\n--- Final Evaluation ---")
with torch.no_grad():
    # FIXED: Used X_test_tensor to evaluate performance against y_test_tensor
    y_test_pred = model.forward_pass(X_test_tensor)
    y_test_pred = (y_test_pred > 0.5).double()  # matches float64/double tensor type
    
    accuracy = (y_test_pred == y_test_tensor).float().mean()
    print(f"Accuracy: {accuracy.item() * 100:.2f}%")

Files Python can see: ['.config', 'drive', 'sample_data']
Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
For epoch number: 0
Y_preds tensor([[1.8510e-01],
        [2.7393e-04],
        [1.7257e-01],
        [9.9998e-01],
        [9.9871e-01],
        [1.0000e+00],
        [5.7787e-03],
        [9.7299e-01],
        [9.0394e-02],
        [9.9024e-01],
        [9.9923e-01],
        [1.0000e+00],
        [3.6053e-03],
        [9.9996e-01],
        [5.0644e-04],
        [3.9619e-02],
        [5.1996e-02],
        [5.3735e-01],
        [1.8052e-04],
        [4.2556e-06],
        [4.9824e-04],
        [8.2827e-02],
        [9.9914e-01],
        [9.8324e-01],
        [1.0000e+00],
        [1.8942e-02],
        [8.8115e-01],
        [3.7338e-03],
        [9.9993e-01],
        [1.0000e+00],
        [3.6587e-04],
        [9.9639e-01],
        [1.0000e+00],
        [9.9265e-01],
        [5.2294e-05],
        [5.75